# Benchmarking RCMAP

In [2]:
# Setup
import arcpy
from arcpy import env
from arcpy.sa import *
import arcgis

# this is a magic command for showing the map in Jupyter notebook
%matplotlib inline

env.workspace = r"\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR"

# Define Analysis Area
analysis_area = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb\CowLakesAssesmentArea_nad83'

# creating bounding box for clip function below
desc = arcpy.Describe(analysis_area).extent
bb = str(desc.XMin) + " " + str(desc.YMin) + " " + str(desc.XMax) + " " + str(desc.YMax)

env.overwriteOutput = True

In [3]:
# Grab RCMAP service for each relevant indicator
# Parameterising this for now in case i want to make it into a tool at some point
indicator_list = 'annual herbaceous' + 'perennial herbaceous' + 'bare ground' + 'shrub cover' + 'tree cover' + 'herbaceous cover' + 'total foliar'
# total foliar cover should be herbaceous + tree + shrub
#indicator_list = 'herbaceous cover'
server = "https://www.mrlc.gov/geoserver/rcmap"
output_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb'

# minimum value is 1985, max value is 2021
year = ["2017","2018","2019","2020", "2021"] # if we want multiple years just need to put this in a loop in cell below and probably create a raster stack for each indicator

# Benchmarks
# could also read in the csv and use a pandas df but this seems easier for now
use_benchmarks = 'TRUE'
annual_herb_benchmarks_80f = RemapRange([[0,1,1],[1,5,2],[5,100,3]])
totalfoliar_benchmarks_80f = RemapRange([[0,49,0],[49,100,1]])
bare_ground_benchmarks_80f = RemapRange([[0,20,1],[20,100,0]])
perennial_annual_ratio_benchmarks_80f = RemapRange([[0,1,0],[1,1000000,1]])

annual_herb_benchmarks_80a = RemapRange([[0,1,1],[1,5,2],[5,100,3]])
totalfoliar_benchmarks_80a = RemapRange([[0,35,0],[35,100,1]])
bare_ground_benchmarks_80a = RemapRange([[0,35,1],[35,100,0]])
perennial_annual_ratio_benchmarks_80a = RemapRange([[0,1,0],[1,1000000,1]])

In [4]:
# # Ideally assign benchmark groups as well
# use benchmark group polygon to select cells and apply approproate benchmark
benchmark_group_poly = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb\LevelIVEcoregion_clip_nad83'
input_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb'

# function to extract by mask, apply benchmarks, then combine again
def benchmark_raster(raster, poly, benchmarks80f, benchmarks80a):
    #harmonize projections
    spatial_ref = arcpy.Describe(raster).spatialReference
    spatial_ref2 = arcpy.Describe(poly).spatialReference
    
    if spatial_ref != spatial_ref2:
        poly = arcpy.management.Project(poly, input_dir + '/poly', spatial_ref)
    
    with arcpy.da.SearchCursor(poly,['SHAPE@','US_L4NAME']) as cursor:
        for row in cursor:
            outmask = arcpy.sa.ExtractByMask(raster, row[0])
            # probably shouldnt hard code benchmark group but its fine for now
            if row[1] == '80f':
                raster_80f = Reclassify(outmask, "Value", benchmarks80f)
            if row[1] == '80a':
                raster_80a = Reclassify(outmask, "Value", benchmarks80f)
            
    arcpy.management.Append(raster_80f, raster_80a, "TEST", None, '', '')
    arcpy.management.Delete(poly)            
    return(raster_80a)

In [30]:
# clipping these to the extent of the analysis area
# need to also specify which years to grab
# there are other indicators but just using these 3 for now

if 'bare ground' in indicator_list:
    for i in year:
        wcs_url = server + "_bare/wcs?coverage=rcmap_bare_ground_"+ i
        bg = arcpy.management.MakeWCSLayer(wcs_url,("bare_ground_"+ i), analysis_area)
        # clip to poly
        bg_output_dir = output_dir + ("/bare_ground_Clip")+ i
        bg_clip = arcpy.management.Clip(bg, bb, bg_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        if use_benchmarks:
            bg_clip_benchmarked = benchmark_raster(bg_clip, benchmark_group_poly, bare_ground_benchmarks_80f, bare_ground_benchmarks_80a)
            bg_clip_benchmarked.save(r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters' + "/bare_ground_benchmarked_"+ i + ".tif")
        arcpy.management.Delete(bg)
        
if 'annual herbaceous' in indicator_list:
    for i in year:
        wcs_url = server + "_anhb/wcs?coverage=rcmap_annual_herbaceous_" + i
        ah = arcpy.management.MakeWCSLayer(wcs_url,("annual_herb"+ i), analysis_area)
        # clip to poly
        ah_output_dir = output_dir + ("/annual_herb_Clip"+ i)
        ah_clip = arcpy.management.Clip(ah, bb, ah_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        if use_benchmarks:
            ah_clip_benchmarked = benchmark_raster(ah_clip, benchmark_group_poly, annual_herb_benchmarks_80f, annual_herb_benchmarks_80a)
            ah_clip_benchmarked.save(r'U:\My Documents\Analysis\OR\Rasters' + "/annual_herb_benchmarked_"+ i + ".tif")
        arcpy.management.Delete(ah)   

        
if 'perennial herbaceous' in indicator_list:
    for i in year :
        wcs_url = server + "_perennial_herbaceous/wcs?coverage=rcmap_perennial_herbaceous_" + i
        ph = arcpy.management.MakeWCSLayer(wcs_url,("perennial_herb"+ i), analysis_area)
        # clip to poly
        ph_output_dir = output_dir + ("/perennial_herb_Clip")+ i
        ph_clip = arcpy.management.Clip(ph, bb, ph_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        arcpy.management.Delete(ph)
        
if 'shrub cover' in indicator_list:
    for i in year :
        wcs_url = server + "_shrub/wcs?coverage=rcmap_shrub_" + i
        sh = arcpy.management.MakeWCSLayer(wcs_url,("shrub"+ i), analysis_area)
        # clip to poly
        sh_output_dir = output_dir + ("/shrub_Clip") + i
        sh_clip = arcpy.management.Clip(sh, bb, sh_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        arcpy.management.Delete(sh)
        
if 'tree cover' in indicator_list:
    for i in year :
        wcs_url = server + "_tree/wcs?coverage=rcmap_tree_" + i
        tr = arcpy.management.MakeWCSLayer(wcs_url,("tree"+ i), analysis_area)
        # clip to poly
        tr_output_dir = output_dir + ("/tree_Clip") + i
        tr_clip = arcpy.management.Clip(tr, bb, tr_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        arcpy.management.Delete(tr)
        
if 'herbaceous cover' in indicator_list:
    for i in year :
        wcs_url = server + "_herb/wcs?coverage=rcmap_herbaceous_" + i
        herb = arcpy.management.MakeWCSLayer(wcs_url,("herb"+ i), analysis_area)
        # clip to poly
        herb_output_dir = output_dir + ("/herb_Clip") + i
        herb_clip = arcpy.management.Clip(herb, bb, herb_output_dir, analysis_area, "256", "ClippingGeometry", "NO_MAINTAIN_EXTENT")
        arcpy.management.Delete(herb)

RuntimeError: ERROR 000875: Output raster: U:\My Documents\Analysis\OR\Rasters\annual_herb_benchmarked_2017.tif's workspace is an invalid output workspace.

In [42]:
# recalced this based on 1-bare ground
if 'total foliar' in indicator_list:
    for i in year:
        base = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb'
        bg_raster = base + '/bare_ground_Clip' + i
        total_foliar = 100 - Raster(bg_raster)
        total_foliar.save(base + '/total_foliar_Clip' + i)
        if use_benchmarks:
            total_foliar_benchmarked = benchmark_raster(total_foliar, benchmark_group_poly, totalfoliar_benchmarks_80f, totalfoliar_benchmarks_80a)
            total_foliar_benchmarked.save(r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters' + "/total_foliar_benchmarked_"+ i + ".tif")
        

In [112]:
# need to calc perennial to annual ratio
# for some reason this isnt working in a loop for doing it for each year
# do I need to reclass no data value first?
output_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb'

for i in year:
    ph_raster = output_dir + "/perennial_herb_Clip" + i 
    ah_raster = output_dir + "/annual_herb_Clip" + i 
    out_divide = Divide(ph_raster, ah_raster)
    out_divide.save(output_dir + "/perennial_annual_ratio_Clip" + i )

In [115]:
# also need to benchmark these
for i in year:
    pa_raster = r'U:\My Documents\Analysis\OR\Remote sensing benchmarks.gdb' + "/perennial_annual_ratio_Clip"+ i
    pa_clip_benchmarked = benchmark_raster(pa_raster, benchmark_group_poly, perennial_annual_ratio_benchmarks_80f, perennial_annual_ratio_benchmarks_80a)
    pa_clip_benchmarked.save(r'U:\My Documents\Analysis\OR\Rasters' + "/perennial_annual_benchmarked_"+ i + ".tif")

In [43]:
# function for getting the 5 year mean 
# for a condition assessment we probably would want a single raster which was the mean of the most recent 5 years of data
#indicator = ["annual_herb", "bare_ground", "perennial_annual_ratio", "total_foliar"]
indicator = ["total_foliar"]
input_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb'
output_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters'
raster_list = []

for j in indicator:
    for i in year:
        raster_list.append(input_dir + "/" + j + "_Clip" + i)
    mean_raster = arcpy.sa.CellStatistics(raster_list, "MEAN", "DATA")
    mean_raster.save(output_dir + "/Mean_" + j + ".tif")
   

In [44]:
 # then benchmark means
output_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters'

#pa_mean_benchmarked = benchmark_raster(output_dir + "/Mean_perennial_annual_ratio.tif", benchmark_group_poly, perennial_annual_ratio_benchmarks_80f, perennial_annual_ratio_benchmarks_80a)
#pa_mean_benchmarked.save(r'U:\My Documents\Analysis\OR\Rasters' + "/Mean_perennial_annual_benchmarked.tif")

#bg_mean_benchmarked = benchmark_raster(output_dir + "/Mean_bare_ground.tif", benchmark_group_poly, bare_ground_benchmarks_80f, bare_ground_benchmarks_80a)
#bg_mean_benchmarked.save(r'U:\My Documents\Analysis\OR\Rasters' + "/Mean_bare_ground_benchmarked.tif")
    
#ah_clip_benchmarked = benchmark_raster(output_dir +"/Mean_annual_herb.tif", benchmark_group_poly, annual_herb_benchmarks_80f, annual_herb_benchmarks_80a)
#ah_clip_benchmarked.save(r'U:\My Documents\Analysis\OR\Rasters' + "/Mean_annual_herb_benchmarked.tif")
    
tf_clip_benchmarked = benchmark_raster(output_dir + "/Mean_total_foliar.tif", benchmark_group_poly, totalfoliar_benchmarks_80f, totalfoliar_benchmarks_80a)
tf_clip_benchmarked.save(r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters\Mean_total_foliar_benchmarked.tif')

In [ ]:
# regression betweennplot data and adjoining raster cells
# find the raster with the closest date
# extract values to points then plot

In [6]:
## Summaries by Allotment
# need to reproject to equal area
def allotment_summary(in_raster, out_raster, allotment_poly, allotment_field, out_table):
    out_proj = 'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",37.5],UNIT["Meter",1.0]]'
    in_proj = arcpy.Describe(in_raster).spatialReference
    arcpy.management.ProjectRaster(in_raster, out_raster, out_proj,"","","","",in_proj)
    arcpy.sa.TabulateArea(allotment_poly, allotment_field, out_raster, "Value", out_table)

In [46]:
#indicator = ["annual_herb", "bare_ground", "perennial_annual", "total_foliar"]
indicator = [ "total_foliar"]
output_dir = r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters'
for i in indicator:
    in_raster = output_dir + "/Mean_" + i + "_benchmarked.tif"
    out_raster = output_dir + "/Mean_" + i + "_benchmarked_aea.tif"
    allotment_summary(in_raster, out_raster, "AllotmentBoundaries", "Allotment Name", output_dir + "/Table_" + i)

In [10]:
# calculating a combined raster from RAP benchmarked rasters
# reclass the rasters to 0 for not meeting and 1 for meeting
# bare ground
#arcpy.ddd.Reclassify(
  #  in_raster=r"RAP Benchmarked\bare_ground_benchmarked_RAP_aea.tif",
  #  reclass_field="Value",
  #  remap="1 1;2 0",
 #   out_raster=r"\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Remote sensing benchmarks.gdb\Reclass_bare1",
 #   missing_values="DATA"
#)

# SUm the rasters (this will reflect teh number of benchmarks that are meeting - 0 to 3)
#output_raster = arcpy.ia.RasterCalculator(
#    expression=' "Reclass_tota1" + "Reclass_pere1" + "Reclass_bare1"'
#)
#output_raster.save(r"\\blm.doi.net\dfs\nr\users\alaurencetraynor\my documents\analysis\or\remote sensing benchmarks.gdb\benchmark_summary_raster")

# calc summary
#allotment_summary("benchmark_summary_raster", r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters\summary_aea',"AllotmentBoundaries","Value", r'\\blm.doi.net\dfs\nr\users\alaurencetraynor\My Documents\Analysis\OR\Rasters\benchmark_summary_Table')

ExecuteError: ERROR 999999: Something unexpected caused the tool to fail. Contact Esri Technical Support (http://esriurl.com/support) to Report a Bug, and refer to the error help for potential solutions or workarounds.
Failed to create raster dataset
Failed to execute (ProjectRaster).
